In [0]:
%sql
CREATE CATALOG IF NOT EXISTS pipeline_1;

CREATE SCHEMA IF NOT EXISTS pipeline_1.bronze;

CREATE VOLUME IF NOT EXISTS pipeline_1.bronze.landing_volume;

CREATE VOLUME IF NOT EXISTS pipeline_1.bronze.checkpoints_volume;

CREATE VOLUME IF NOT EXISTS pipeline_1.bronze.source_volume;


In [0]:
LANDING_VOLUME = "/Volumes/pipeline_1/bronze/landing_volume"
CHECKPOINT_PATH = "/Volumes/pipeline_1/bronze/checkpoints_volume/bronze_checkpoint"
SCHEMA_PATH = "/Volumes/pipeline_1/bronze/checkpoints_volume/bronze_schema"
BRONZE_TABLE = "pipeline_1.bronze.raw_events"

In [0]:
from pyspark.sql.functions import current_timestamp

# Читаем данные из landing_volume, автоматически определяя тип данных
df = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "json")
    .option("cloudfiles.schemaLocation", SCHEMA_PATH)
    .option("cloudfiles.inferColumnTypes", "true")
    .load(LANDING_VOLUME)
    .withColumn("ingestion_ts", current_timestamp())

)

# Записываем данные в таблицу bronze.raw_events
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)